# 🧘 Task Baba: model server on Google Colab

This notebook keeps **Task Baba's language model in your Google Drive** and runs it on a Colab GPU, so nothing
heavy lives on your Mac.

**What it does**
1. Mounts your Drive and keeps the model in `MyDrive/TaskBaba/models` (downloaded once from Hugging Face at a pinned
   revision, resumable, SHA-256 verified before it becomes active; the previous model is kept in `backup/` for rollback).
2. Downloads a pinned, checksum-verified build of `llama.cpp` and starts `llama-server` on `127.0.0.1` only.
3. Starts a small auth/limits proxy (API key, 16 KB body limit, ≤200 tokens, rate limit, no prompt logging).
4. Opens a Cloudflare *quick tunnel* to the proxy and prints an `https://…trycloudflare.com` URL and an API key.
   Paste both into **Task Baba → AI & Model**.

**Please read: honest limits**
* **Drive only stores the file; Colab does the computing.** When this notebook stops, Task Baba falls back to its
  built-in lines automatically.
* Free Colab sessions disconnect when idle and have a maximum lifetime (often around 12 hours), and GPUs are not
  always available. Colab is designed for interactive use, not as an always-on server, so check the current
  [Colab FAQ / terms](https://research.google.com/colaboratory/faq.html) before relying on it.
* Cloudflare quick tunnels are meant for testing: no uptime guarantee, and **the URL changes every run**.
* Requests travel through Cloudflare and Google infrastructure. Task Baba only sends what you consented to in the app,
  and this proxy never logs prompts or replies.
* For dependable hosting, see `cloud/server/` in the repository (a paid server you would create yourself).

Use **Runtime → Change runtime type → T4 GPU** and then **Runtime → Run all**.

In [ ]:
# @title 1 · Settings { display-mode: "form" }
MODEL_CHOICE = "qwen3-4b-q4km"  # @param ["qwen3-4b-q4km", "qwen3-1.7b-q8"]
DRIVE_DIR = "/content/drive/MyDrive/TaskBaba/models"  # @param {type:"string"}
# Keep the same API key across sessions by saving it in Drive (TaskBaba/api_key.txt). Off = new key each run.
SAVE_KEY_TO_DRIVE = False  # @param {type:"boolean"}
CONTEXT_SIZE = 2048  # @param {type:"integer"}
MAX_TOKENS = 200  # hard cap, also enforced by the proxy

MODELS = {
    "qwen3-4b-q4km": {
        "repo": "Qwen/Qwen3-4B-GGUF", "revision": "bc640142c66e1fdd12af0bd68f40445458f3869b",
        "file": "Qwen3-4B-Q4_K_M.gguf", "size": 2497280256,
        "sha256": "7485fe6f11af29433bc51cab58009521f205840f5b4ae3a32fa7f92e8534fdf5", "license": "Apache-2.0",
    },
    "qwen3-1.7b-q8": {
        "repo": "Qwen/Qwen3-1.7B-GGUF", "revision": "90862c4b9d2787eaed51d12237eafdfe7c5f6077",
        "file": "Qwen3-1.7B-Q8_0.gguf", "size": 1834426016,
        "sha256": "061b54daade076b5d3362dac252678d17da8c68f07560be70818cace6590cb1a", "license": "Apache-2.0",
    },
}

# Pinned third-party binaries (GitHub release assets) with their published SHA-256 digests.
LLAMA_BUILD = "b11355"
LLAMA_ASSETS = {
    "cuda": ("llama-b11355-bin-ubuntu-cuda-12.8-x64.tar.gz", "cb0da078ea93c64a57138cb811df26099195bd8db7eef2fefd0661fc3e982d59"),
    "cudart": ("cudart-llama-b11355-bin-ubuntu-cuda-12.8-x64.tar.gz", "619f42ab920777ceb05e96a840cb6941a1c3963118ad564de017f1ec3ef51f0f"),
    "cpu": ("llama-b11355-bin-ubuntu-x64.tar.gz", "4a21de9ce9f0cbb9cd2bb81d241919c8ab62c498e867a57bddbcf16cfcef1dbf"),
}
CLOUDFLARED_VERSION = "2026.10.0"
CLOUDFLARED_SHA256 = "d33ff2d14475178d2012c2c56beba87389ac5ded27649519f198a7d3134a99db"

spec = MODELS[MODEL_CHOICE]
print(f"Model: {spec['file']} ({spec['size']/1e9:.2f} GB, {spec['license']}) from {spec['repo']}@{spec['revision'][:12]}")

In [ ]:
# @title 2 · Mount Google Drive
import os
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(os.path.join(DRIVE_DIR, "backup"), exist_ok=True)
print("Model folder:", DRIVE_DIR)

In [ ]:
# @title 3 · Helpers (download with resume, verify)
import hashlib, json, os, shutil, time, urllib.request, urllib.error

def sha256_file(path, chunk=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def download(url, dest, expected_size=None, retries=8):
    """Download to dest with HTTP Range resume. Returns when the file has expected_size bytes."""
    for attempt in range(retries):
        have = os.path.getsize(dest) if os.path.exists(dest) else 0
        if expected_size and have == expected_size:
            return
        if expected_size and have > expected_size:
            os.remove(dest); have = 0
        req = urllib.request.Request(url, headers={"User-Agent": "TaskBaba-Colab/1.0"})
        if have:
            req.add_header("Range", f"bytes={have}-")
        try:
            with urllib.request.urlopen(req, timeout=60) as r:
                if have and r.status != 206:      # server ignored Range: start over
                    have = 0
                mode = "ab" if have else "wb"
                total = expected_size or int(r.headers.get("Content-Length", 0)) + have
                last = time.time()
                with open(dest, mode) as f:
                    while True:
                        b = r.read(4 * 1024 * 1024)
                        if not b:
                            break
                        f.write(b); have += len(b)
                        if time.time() - last > 10:
                            print(f"  {have/1e9:.2f} / {total/1e9:.2f} GB", flush=True); last = time.time()
            if not expected_size or os.path.getsize(dest) == expected_size:
                return
        except (urllib.error.URLError, TimeoutError, ConnectionError) as e:
            print(f"  interrupted ({e.__class__.__name__}); resuming in {2 ** attempt}s…")
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Download did not complete after {retries} attempts: {url}")

def verify_asset(path, sha):
    got = sha256_file(path)
    if got != sha:
        os.remove(path)
        raise RuntimeError(f"Checksum mismatch for {os.path.basename(path)}: expected {sha}, got {got}. File removed.")

def marker_path(path):
    return path + ".verified.json"

def is_verified(path, spec):
    m = marker_path(path)
    if not (os.path.exists(path) and os.path.exists(m)):
        return False
    try:
        info = json.load(open(m))
    except Exception:
        return False
    st = os.stat(path)
    return info.get("sha256") == spec["sha256"] and info.get("size") == st.st_size == spec["size"] and info.get("mtime") == int(st.st_mtime)

def write_marker(path, spec):
    st = os.stat(path)
    json.dump({"sha256": spec["sha256"], "size": st.st_size, "mtime": int(st.st_mtime),
               "repo": spec["repo"], "revision": spec["revision"], "license": spec["license"]}, open(marker_path(path), "w"))
print("Helpers ready.")

In [ ]:
# @title 4 · Install or verify the model in Drive
active = os.path.join(DRIVE_DIR, spec["file"])
part = active + ".part"
backup_dir = os.path.join(DRIVE_DIR, "backup")

def move_to_backup(path):
    dest = os.path.join(backup_dir, os.path.basename(path))
    if os.path.exists(dest):
        os.remove(dest)
    shutil.move(path, dest)
    if os.path.exists(marker_path(path)):
        shutil.move(marker_path(path), marker_path(dest))
    print("  kept previous copy in backup/:", os.path.basename(path))

if is_verified(active, spec):
    print("✅ Model already in Drive and verified:", spec["file"])
else:
    if os.path.exists(active):
        print("Checking the existing file (takes a minute for a few GB)…")
        if os.path.getsize(active) == spec["size"] and sha256_file(active) == spec["sha256"]:
            write_marker(active, spec)
            print("✅ Existing file verified.")
        else:
            bad = active + ".corrupt"
            os.replace(active, bad)
            print("⚠️ Existing file failed verification; moved aside as", os.path.basename(bad))
    if not is_verified(active, spec):
        url = f"https://huggingface.co/{spec['repo']}/resolve/{spec['revision']}/{spec['file']}"
        print(f"Downloading {spec['file']} into Drive (resumable)…")
        download(url, part, spec["size"])
        print("Verifying SHA-256…")
        got = sha256_file(part)
        if got != spec["sha256"]:
            os.remove(part)
            raise RuntimeError(f"Checksum mismatch (expected {spec['sha256']}, got {got}). The partial file was deleted; run this cell again.")
        # Keep any other catalog model that was active as the rollback copy.
        for other in MODELS.values():
            p = os.path.join(DRIVE_DIR, other["file"])
            if other["file"] != spec["file"] and os.path.exists(p):
                move_to_backup(p)
        os.replace(part, active)
        write_marker(active, spec)
        print("✅ Model installed and verified in Drive:", active)

In [ ]:
# @title 5 · (Optional) Restore the previous model from backup/
RESTORE = False  # @param {type:"boolean"}
if RESTORE:
    backups = [m for m in MODELS.values() if os.path.exists(os.path.join(backup_dir, m["file"]))]
    if not backups:
        print("No backup found.")
    else:
        b = backups[0]
        src = os.path.join(backup_dir, b["file"])
        print("Verifying backup before restoring…")
        if os.path.getsize(src) != b["size"] or sha256_file(src) != b["sha256"]:
            raise RuntimeError("The backup failed verification; not restoring it.")
        for m in MODELS.values():
            p = os.path.join(DRIVE_DIR, m["file"])
            if os.path.exists(p) and m["file"] != b["file"]:
                move_to_backup(p)
        dest = os.path.join(DRIVE_DIR, b["file"])
        shutil.move(src, dest)
        write_marker(dest, b)
        MODEL_CHOICE = [k for k, v in MODELS.items() if v is b][0]
        spec = b; active = dest
        print("✅ Restored", b["file"], "as the active model. Continue with cell 6.")
else:
    print("Skipped (tick RESTORE to roll back).")

In [ ]:
# @title 6 · (Optional) Remove model files from Drive
CONFIRM_REMOVE = False  # @param {type:"boolean"}
ALSO_REMOVE_BACKUP = False  # @param {type:"boolean"}
if CONFIRM_REMOVE:
    for p in [active, marker_path(active), part]:
        if os.path.exists(p):
            os.remove(p); print("Removed", p)
    if ALSO_REMOVE_BACKUP:
        for m in MODELS.values():
            for p in [os.path.join(backup_dir, m["file"]), marker_path(os.path.join(backup_dir, m["file"]))]:
                if os.path.exists(p):
                    os.remove(p); print("Removed", p)
    print("Done. Files go to Drive's trash, which Google empties after 30 days.")
else:
    print("Skipped (tick CONFIRM_REMOVE to delete).")

In [ ]:
# @title 7 · Install llama.cpp (pinned, checksum-verified)
import subprocess
gpu = subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print("GPU runtime:" if gpu else "No GPU found; using the CPU build (slow, use the 1.7B model).")
base = f"https://github.com/ggml-org/llama.cpp/releases/download/{LLAMA_BUILD}/"
name, sha = LLAMA_ASSETS["cuda" if gpu else "cpu"]
os.makedirs("/content/llama", exist_ok=True)
tgz = f"/content/{name}"
if not os.path.exists(f"/content/llama/llama-{LLAMA_BUILD}/llama-server"):
    download(base + name, tgz)
    verify_asset(tgz, sha)
    subprocess.run(["tar", "-xzf", tgz, "-C", "/content/llama"], check=True)
LLAMA_DIR = f"/content/llama/llama-{LLAMA_BUILD}"
SERVER = os.path.join(LLAMA_DIR, "llama-server")
env = dict(os.environ, LD_LIBRARY_PATH=LLAMA_DIR + ":" + os.environ.get("LD_LIBRARY_PATH", ""))
v = subprocess.run([SERVER, "--version"], capture_output=True, text=True, env=env)
if v.returncode != 0 and gpu:
    print("Fetching the matching CUDA runtime libraries…")
    cname, csha = LLAMA_ASSETS["cudart"]
    ctgz = f"/content/{cname}"
    download(base + cname, ctgz)
    verify_asset(ctgz, csha)
    subprocess.run(["tar", "-xzf", ctgz, "-C", "/content/llama"], check=True)
    v = subprocess.run([SERVER, "--version"], capture_output=True, text=True, env=env)
if v.returncode != 0:
    raise RuntimeError("llama-server failed to start: " + (v.stderr or v.stdout)[-500:])
print("✅", (v.stdout or v.stderr).strip().splitlines()[-1])

In [ ]:
# @title 8 · Start llama-server (localhost only)
import secrets, urllib.request
local_model = "/content/model.gguf"
free = shutil.disk_usage("/content").free
if not (os.path.exists(local_model) and os.path.getsize(local_model) == spec["size"]):
    if free > spec["size"] * 1.2:
        print("Copying the model from Drive to the Colab VM's disk for faster loading…")
        shutil.copyfile(active, local_model)
    else:
        local_model = active  # not enough local disk: load straight from Drive (slower)
UPSTREAM_KEY = secrets.token_urlsafe(32)
try:
    llama_proc.terminate()
except NameError:
    pass
llama_log = open("/content/llama-server.log", "w")
llama_proc = subprocess.Popen([
    SERVER, "-m", local_model, "--host", "127.0.0.1", "--port", "8081",
    "--api-key", UPSTREAM_KEY, "-c", str(CONTEXT_SIZE), "-np", "1", "-n", str(MAX_TOKENS),
    "-ngl", "99" if gpu else "0", "--no-webui", "--no-slots", "--reasoning", "off", "--timeout", "120",
], stdout=llama_log, stderr=subprocess.STDOUT, env=env)

for i in range(180):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8081/health", timeout=2) as r:
            if r.status == 200:
                print("✅ Model loaded."); break
    except Exception:
        pass
    if llama_proc.poll() is not None:
        raise RuntimeError("llama-server exited; see /content/llama-server.log")
    time.sleep(2)
else:
    raise RuntimeError("Model did not load within 6 minutes; see /content/llama-server.log")

In [ ]:
# @title 9 · Start the Task Baba auth proxy
PROXY_SOURCE = r'''
#!/usr/bin/env python3
"""Task Baba inference proxy.

A small, dependency-free gatekeeper in front of a llama.cpp server bound to 127.0.0.1.
Only this proxy is exposed (through a Cloudflare tunnel in Colab, or a load balancer elsewhere).

It enforces:
  * Bearer API key (constant-time comparison) on every data endpoint
  * request body <= 16 KB, <= 8 messages, max_tokens <= 200, non-streaming, n = 1
  * a whitelist of request fields (everything else is dropped)
  * per-minute rate limits (overall and for failed auth attempts per client)
  * a small concurrency cap with a bounded wait
  * no logging of prompts, responses, headers or keys: only method, path, status and latency

Endpoints:
  GET  /livez                  -> 200 "ok" (no auth, no data; for uptime checks)
  GET  /health                 -> 200 when the model is loaded, 503 while loading (auth required)
  POST /v1/chat/completions    -> forwarded to llama.cpp (auth required)

Configuration (environment):
  TB_API_KEY        required; the key clients must send
  TB_UPSTREAM       default http://127.0.0.1:8081
  TB_UPSTREAM_KEY   key for the llama.cpp server's own --api-key (defense in depth)
  TB_LISTEN_HOST    default 127.0.0.1
  TB_LISTEN_PORT    default 8080
  TB_RATE_PER_MIN   default 30
  TB_MAX_CONCURRENT default 2
"""

import hmac
import json
import os
import sys
import threading
import time
import urllib.error
import urllib.request
from collections import defaultdict, deque
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

MAX_BODY = 16 * 1024
MAX_MESSAGES = 8
MAX_CONTENT_CHARS = 8000
MAX_TOKENS_CAP = 200
ALLOWED_FIELDS = {"messages", "max_tokens", "temperature", "top_p", "chat_template_kwargs", "model"}
ALLOWED_ROLES = {"system", "user", "assistant"}


class Limits:
    def __init__(self, per_minute: int, auth_failures_per_minute: int = 10):
        self.per_minute = per_minute
        self.auth_failures_per_minute = auth_failures_per_minute
        self._lock = threading.Lock()
        self._requests = deque()
        self._failures = defaultdict(deque)

    @staticmethod
    def _prune(q, now):
        while q and now - q[0] > 60:
            q.popleft()

    def allow_request(self) -> bool:
        now = time.monotonic()
        with self._lock:
            self._prune(self._requests, now)
            if len(self._requests) >= self.per_minute:
                return False
            self._requests.append(now)
            return True

    def auth_blocked(self, client: str) -> bool:
        now = time.monotonic()
        with self._lock:
            q = self._failures[client]
            self._prune(q, now)
            return len(q) >= self.auth_failures_per_minute

    def record_auth_failure(self, client: str):
        with self._lock:
            self._failures[client].append(time.monotonic())


def sanitize_request(raw: bytes):
    """Return (payload, error). The payload only contains whitelisted, bounded fields."""
    try:
        body = json.loads(raw.decode("utf-8"))
    except (UnicodeDecodeError, json.JSONDecodeError):
        return None, "body must be JSON"
    if not isinstance(body, dict):
        return None, "body must be a JSON object"
    messages = body.get("messages")
    if not isinstance(messages, list) or not (1 <= len(messages) <= MAX_MESSAGES):
        return None, f"messages must be a list of 1-{MAX_MESSAGES} items"
    total = 0
    clean_messages = []
    for m in messages:
        if not isinstance(m, dict) or m.get("role") not in ALLOWED_ROLES or not isinstance(m.get("content"), str):
            return None, "each message needs a role and string content"
        total += len(m["content"])
        clean_messages.append({"role": m["role"], "content": m["content"]})
    if total > MAX_CONTENT_CHARS:
        return None, "messages too long"
    payload = {k: v for k, v in body.items() if k in ALLOWED_FIELDS}
    payload["messages"] = clean_messages
    try:
        max_tokens = int(payload.get("max_tokens", 80))
    except (TypeError, ValueError):
        return None, "max_tokens must be an integer"
    payload["max_tokens"] = max(1, min(max_tokens, MAX_TOKENS_CAP))
    try:
        payload["temperature"] = max(0.0, min(float(payload.get("temperature", 0.8)), 1.5))
    except (TypeError, ValueError):
        return None, "temperature must be a number"
    if "top_p" in payload:
        try:
            payload["top_p"] = max(0.0, min(float(payload["top_p"]), 1.0))
        except (TypeError, ValueError):
            payload.pop("top_p")
    kwargs = payload.get("chat_template_kwargs")
    payload["chat_template_kwargs"] = {"enable_thinking": bool(kwargs.get("enable_thinking", False))} if isinstance(kwargs, dict) else {"enable_thinking": False}
    payload["stream"] = False
    payload["n"] = 1
    return payload, None


def make_handler(api_key: str, upstream: str, upstream_key: str, limits: Limits, slots: threading.BoundedSemaphore):
    expected = ("Bearer " + api_key).encode()

    class Handler(BaseHTTPRequestHandler):
        server_version = "TaskBabaProxy/1.0"
        sys_version = ""
        protocol_version = "HTTP/1.1"

        # Only method, path (no query), status and latency are logged.
        def log_message(self, fmt, *args):
            pass

        def _log(self, status, started):
            path = self.path.split("?", 1)[0]
            sys.stderr.write(f"{self.command} {path} {status} {int((time.monotonic() - started) * 1000)}ms\n")

        def _client(self):
            return self.headers.get("CF-Connecting-IP") or self.client_address[0]

        def _send(self, status, obj=None, text=None, started=None):
            data = text.encode() if text is not None else json.dumps(obj or {}).encode()
            self.send_response(status)
            self.send_header("Content-Type", "text/plain" if text is not None else "application/json")
            self.send_header("Content-Length", str(len(data)))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            self.wfile.write(data)
            if started is not None:
                self._log(status, started)

        def _authorized(self, started) -> bool:
            client = self._client()
            if limits.auth_blocked(client):
                self._send(429, {"error": "too many failed attempts"}, started=started)
                return False
            got = (self.headers.get("Authorization") or "").encode()
            if not hmac.compare_digest(got, expected):
                limits.record_auth_failure(client)
                self._send(401, {"error": "unauthorized"}, started=started)
                return False
            return True

        def _upstream(self, method, path, body=None, timeout=90):
            req = urllib.request.Request(upstream + path, data=body, method=method)
            req.add_header("Content-Type", "application/json")
            if upstream_key:
                req.add_header("Authorization", "Bearer " + upstream_key)
            return urllib.request.urlopen(req, timeout=timeout)

        def do_GET(self):
            started = time.monotonic()
            path = self.path.split("?", 1)[0]
            if path == "/livez":
                return self._send(200, text="ok", started=started)
            if path != "/health":
                return self._send(404, {"error": "not found"}, started=started)
            if not self._authorized(started):
                return
            try:
                with self._upstream("GET", "/health", timeout=5) as r:
                    status = r.status
            except urllib.error.HTTPError as e:
                status = e.code
            except Exception:
                status = 502
            if status == 200:
                self._send(200, {"status": "ok"}, started=started)
            elif status == 503:
                self._send(503, {"status": "loading"}, started=started)
            else:
                self._send(502, {"status": "upstream unavailable"}, started=started)

        def do_POST(self):
            started = time.monotonic()
            path = self.path.split("?", 1)[0]
            if path != "/v1/chat/completions":
                return self._send(404, {"error": "not found"}, started=started)
            if not self._authorized(started):
                return
            try:
                length = int(self.headers.get("Content-Length") or "-1")
            except ValueError:
                length = -1
            if length < 0:
                return self._send(411, {"error": "content-length required"}, started=started)
            if length > MAX_BODY:
                return self._send(413, {"error": "request too large"}, started=started)
            raw = self.rfile.read(length)
            payload, err = sanitize_request(raw)
            if err:
                return self._send(400, {"error": err}, started=started)
            if not limits.allow_request():
                return self._send(429, {"error": "rate limit"}, started=started)
            if not slots.acquire(timeout=20):
                return self._send(503, {"error": "busy"}, started=started)
            try:
                with self._upstream("POST", "/v1/chat/completions", json.dumps(payload).encode()) as r:
                    data = r.read(256 * 1024)
                    status = r.status
            except urllib.error.HTTPError as e:
                status = 503 if e.code == 503 else 502
                data = json.dumps({"error": "upstream error"}).encode()
            except Exception:
                status, data = 504, json.dumps({"error": "upstream timeout"}).encode()
            finally:
                slots.release()
            self.send_response(status)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(data)))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            self.wfile.write(data)
            self._log(status, started)

    return Handler


def main():
    api_key = os.environ.get("TB_API_KEY", "")
    if len(api_key) < 24:
        sys.exit("TB_API_KEY must be set to a long random value")
    upstream = os.environ.get("TB_UPSTREAM", "http://127.0.0.1:8081").rstrip("/")
    upstream_key = os.environ.get("TB_UPSTREAM_KEY", "")
    host = os.environ.get("TB_LISTEN_HOST", "127.0.0.1")
    port = int(os.environ.get("TB_LISTEN_PORT", "8080"))
    limits = Limits(int(os.environ.get("TB_RATE_PER_MIN", "30")))
    slots = threading.BoundedSemaphore(int(os.environ.get("TB_MAX_CONCURRENT", "2")))
    server = ThreadingHTTPServer((host, port), make_handler(api_key, upstream, upstream_key, limits, slots))
    server.daemon_threads = True
    sys.stderr.write(f"Task Baba proxy listening on {host}:{port} -> {upstream}\n")
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        pass


if __name__ == "__main__":
    main()

'''
open("/content/taskbaba_proxy.py", "w").write(PROXY_SOURCE)

key_file = "/content/drive/MyDrive/TaskBaba/api_key.txt"
API_KEY = None
try:
    from google.colab import userdata          # optional Colab secret named TASKBABA_API_KEY
    API_KEY = userdata.get("TASKBABA_API_KEY")
except Exception:
    pass
if not API_KEY and SAVE_KEY_TO_DRIVE and os.path.exists(key_file):
    API_KEY = open(key_file).read().strip()
if not API_KEY:
    API_KEY = secrets.token_urlsafe(32)
    if SAVE_KEY_TO_DRIVE:
        with open(key_file, "w") as f:
            f.write(API_KEY)
        os.chmod(key_file, 0o600)

try:
    proxy_proc.terminate()
except NameError:
    pass
proxy_proc = subprocess.Popen(
    ["python3", "/content/taskbaba_proxy.py"],
    env=dict(os.environ, TB_API_KEY=API_KEY, TB_UPSTREAM="http://127.0.0.1:8081", TB_UPSTREAM_KEY=UPSTREAM_KEY,
             TB_LISTEN_HOST="127.0.0.1", TB_LISTEN_PORT="8080", TB_RATE_PER_MIN="30", TB_MAX_CONCURRENT="2"),
    stdout=open("/content/proxy.log", "w"), stderr=subprocess.STDOUT,
)
time.sleep(1.5)
with urllib.request.urlopen("http://127.0.0.1:8080/livez", timeout=5) as r:
    assert r.status == 200
print("✅ Proxy running on 127.0.0.1:8080 (logs: method, path, status, latency only).")

In [ ]:
# @title 10 · Open the HTTPS tunnel and print your endpoint
import re
cf = "/content/cloudflared"
if not os.path.exists(cf):
    download(f"https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64", cf)
    verify_asset(cf, CLOUDFLARED_SHA256)
    os.chmod(cf, 0o755)
try:
    tunnel_proc.terminate()
except NameError:
    pass
tunnel_log = "/content/cloudflared.log"
tunnel_proc = subprocess.Popen([cf, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8080"],
                               stdout=open(tunnel_log, "w"), stderr=subprocess.STDOUT)
ENDPOINT = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(tunnel_log).read())
    if m:
        ENDPOINT = m.group(0); break
if not ENDPOINT:
    raise RuntimeError("Tunnel did not start; see /content/cloudflared.log")
print("Paste these into Task Baba → AI & Model:\n")
print("  Endpoint URL:", ENDPOINT)
print("  API key:     ", API_KEY)
print("\nKeep this tab open. The URL stops working when the session ends.")

In [ ]:
# @title 11 · Self-test through the tunnel
import json
time.sleep(3)
hdr = {"Authorization": "Bearer " + API_KEY, "Content-Type": "application/json"}
with urllib.request.urlopen(urllib.request.Request(ENDPOINT + "/health", headers=hdr), timeout=20) as r:
    print("health:", r.status)
body = json.dumps({"messages": [
    {"role": "system", "content": "You are Baba, a witty, kind monk. One short sentence."},
    {"role": "user", "content": "Remind me to drink water. /no_think"}],
    "max_tokens": 60, "chat_template_kwargs": {"enable_thinking": False}}).encode()
t = time.time()
with urllib.request.urlopen(urllib.request.Request(ENDPOINT + "/v1/chat/completions", data=body, headers=hdr), timeout=60) as r:
    reply = json.load(r)["choices"][0]["message"]["content"]
print(f"Baba says ({time.time() - t:.1f}s):", reply.strip())

In [ ]:
# @title 12 · Status / stop
STOP = False  # @param {type:"boolean"}
for name in ["tunnel_proc", "proxy_proc", "llama_proc"]:
    p = globals().get(name)
    state = "not started" if p is None else ("running" if p.poll() is None else f"exited ({p.returncode})")
    print(f"{name:12s} {state}")
    if STOP and p is not None and p.poll() is None:
        p.terminate(); print("  stopped")